# 8. Current business/source-category workspace audit

## tl;dr

This runnable audit reproduces the current app's **individual-business** decisions: choose a canonical city, parent category, exact source tag and dates; rank source categories and competitors; compare business profiles and activity against shared weather distributions; inspect supported existing-business projections after recorded coverage ends.

Every result comes from the current imported database and verified local artifacts. No old ZIP rankings, embedded WBS scores, cached weather effects, or area-model metrics are reused. Run all cells to obtain current values; the notebook intentionally contains no frozen conclusion about the best niche or a new-store address.

## Context & Methods

This is a read-only, reproducible audit for a business analyst, not a deployment or project-completion certificate. The analytical app workflows include [Location map](../docs/business-location-workspace.md), Compare businesses, and [Business analysis](../docs/business-analysis.md); the navigation inventory below also captures current informational pages. The map's weather/activity chart covers **all matching businesses**; its top-business map limit does not limit totals. Compare businesses overlays up to four individual activity lines and one shared weather distribution, followed by a snapshot/profile matrix.

### Key assumptions

- Exact source-category tags overlap. A listing contributes its whole activity count to every matching tag; tag totals are not additive and do not allocate activity to a service.
- `business_id` is the ranking/comparison identity. Snapshot name, categories, stars, review count and open status do not establish historical membership, opening dates or operational status.
- Covered calendar days without buckets mean zero **Recorded check-ins**. Outside coverage and failed reads remain unknown. Check-ins are an activity proxy, not customers, revenue or market size.
- Station measurements are not measurements at each shop. Weather boxes use unique mapped station/date values, or optional city-airport hourly temperatures. Rain is daily accepted station totals; it is not summed across stations. Station report windows and assumed source-local activity dates are not verified to align.
- Weather association is descriptive. Observed target-day weather is not advance forecast weather. Current app projections use individual-business HGB with an explicit weather scenario. The default scenario uses training-station/month averages, not live forecast weather. HGB serving does not establish an accuracy win, a causal weather effect, or a new store's performance.
- Forecast support requires a compatible artifact matching source import, code, lockfile and payload hashes. Missing support is unavailable, never a zero or an inferred new-shop estimate.

## Data

### 1. Setup and visible reader inputs

Start the populated local database using the [setup guide](../docs/agents/local-development.md). Run from the repository root with the project kernel and `DATABASE_URL` supplied through the environment:

```sh
uv run --frozen --group notebooks --env-file .env jupyter nbconvert --execute --to notebook --output-dir /private/tmp/sitesense-audit notebooks/08_project_state_usecase_audit.ipynb
```

Database connections are forced read-only with a bounded statement timeout. The notebook does not migrate, import, download, train, publish models or modify Streamlit session state. Outputs are written only under ignored `data/processed/business_workspace_audit/run_*` directories.

In [ ]:
import ast
import hashlib
import json
import os
import platform
import sys
from dataclasses import asdict, replace
from datetime import UTC, date, datetime
from importlib.metadata import version
from pathlib import Path

ROOT = next(
    (
        path
        for path in (Path.cwd(), *Path.cwd().parents)
        if (path / "pyproject.toml").is_file() and (path / "src/sitesense").is_dir()
    ),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Run inside the SiteSense repository.")
sys.path.insert(0, str(ROOT / "src"))
os.environ.setdefault("MPLCONFIGDIR", "/private/tmp/sitesense-notebook-mpl")
# Every repository helper opens its own connection; PGOPTIONS applies these guards to all.
os.environ["PGOPTIONS"] = (
    os.environ.get("PGOPTIONS", "")
    + " -c default_transaction_read_only=on -c statement_timeout=60000"
)
if not os.environ.get("DATABASE_URL", "").strip():
    raise RuntimeError(
        "Supply DATABASE_URL through --env-file .env; no credential is printed here."
    )

In [ ]:
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from sitesense import business_analysis, business_hgb, repository
from sitesense.database import connect
from sitesense.hourly_weather import chart_rows as hourly_chart_rows
from sitesense.location_weather import load_daily_station_weather
from sitesense.models import BUSINESS_CATEGORY_LABELS, CITY_SCOPES, Filters
from sitesense.weather_charts import (
    activity_period_rows,
    calendar_grain,
    weather_boxplot_display_rows,
    weather_boxplot_rows,
)

CITY, STATE = "Philadelphia", "PA"
PARENT_CATEGORY = "Restaurants"  # Displayed as Restaurant; Day Spas is displayed as Spa.
SOURCE_CATEGORY = "Vietnamese"  # Exact co-tag; None means all source categories.
START_DATE, END_DATE = date(2019, 1, 1), date(2019, 12, 31)
FUTURE_START, FUTURE_END = date(2026, 11, 1), date(2026, 11, 30)
WEATHER_SCENARIO = None  # Or (15.0, 25.0, 0.0): low °C, high °C, rain mm/day.
MODEL_DIRECTORY = ROOT / "models/business_hgb"
COMPARE_BUSINESS_IDS = ()  # Optional exact IDs; otherwise choose the two leading competitors.
TOP_N = 10
WEATHER_VARIABLE = "Temperature"  # Temperature or Precipitation.

assert (CITY, STATE) in {(scope.city, scope.state) for scope in CITY_SCOPES}
assert PARENT_CATEGORY in BUSINESS_CATEGORY_LABELS and SOURCE_CATEGORY != PARENT_CATEGORY
assert START_DATE <= END_DATE and FUTURE_START <= FUTURE_END
if WEATHER_SCENARIO is not None:
    business_hgb.validate_weather(WEATHER_SCENARIO)
assert WEATHER_VARIABLE in {"Temperature", "Precipitation"} and TOP_N > 0
RUN_DIR = (
    ROOT
    / "data/processed/business_workspace_audit"
    / datetime.now(UTC).strftime("run_%Y%m%dT%H%M%S%fZ")
)
RUN_DIR.mkdir(parents=True, exist_ok=False)
ARTIFACTS = []


def sha256(path):
    with Path(path).open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()


def save_table(frame, name):
    frame.to_csv(RUN_DIR / name, index=False)
    ARTIFACTS.append(name)


def save_figure(figure, name):
    figure.savefig(RUN_DIR / name, dpi=140, bbox_inches="tight")
    ARTIFACTS.append(name)
    plt.show()
    plt.close(figure)


SOURCE_PATHS = sorted(
    {
        "uv.lock",
        "notebooks/08_project_state_usecase_audit.ipynb",
        *[str(path.relative_to(ROOT)) for path in (ROOT / "src/sitesense").rglob("*.py")],
    }
)
SOURCE_HASHES_AT_START = {name: sha256(ROOT / name) for name in SOURCE_PATHS}
print(
    {
        "scope": (
            f"{CITY}/{STATE} · {PARENT_CATEGORY} · {SOURCE_CATEGORY or 'All source categories'}"
        ),
        "recorded_dates": [START_DATE.isoformat(), END_DATE.isoformat()],
        "future_dates": [FUTURE_START.isoformat(), FUTURE_END.isoformat()],
        "output": str(RUN_DIR.relative_to(ROOT)),
    }
)

### 2. Current imported coverage and read-only safety

The activity import's manifest supplies the coverage contract. The audit stops on a failed read instead of filling unknown activity with zero. It records the import revision before analysis and checks it again at handoff; data from a concurrent reimport are not silently combined.

In [ ]:
try:
    with connect() as connection:
        readonly = connection.execute("SHOW default_transaction_read_only").fetchone()[0]
        timeout = connection.execute("SHOW statement_timeout").fetchone()[0]
    assert readonly == "on"
    import_revision = repository.get_import_revision()
    hourly_revision = repository.get_hourly_weather_revision()
    catalog = repository.load_catalog(CITY, STATE, PARENT_CATEGORY)
except Exception:
    raise RuntimeError(
        "Cannot read the completed local import. Check the database setup and supplied environment."
    ) from None
coverage = catalog.coverage
covered_start, covered_end = max(START_DATE, coverage.start_date), min(END_DATE, coverage.end_date)
covered = covered_start <= covered_end
if FUTURE_START <= coverage.end_date:
    raise ValueError("Future planning must begin strictly after imported activity coverage ends.")
filters = Filters(
    CITY, STATE, PARENT_CATEGORY, START_DATE, END_DATE, source_category=SOURCE_CATEGORY
)
effective = replace(filters, start_date=covered_start, end_date=covered_end) if covered else None
print(
    {
        "read_only": readonly,
        "statement_timeout": timeout,
        "import_revision": import_revision,
        "hourly_revision": hourly_revision,
        "coverage_start": coverage.start_date.isoformat(),
        "coverage_end": coverage.end_date.isoformat(),
        "timestamp_policy": coverage.timestamp_policy,
        "selected_covered_days": (covered_end - covered_start).days + 1 if covered else 0,
        "parent_catalog_businesses": len(catalog.businesses),
    }
)
navigation = ast.parse((ROOT / "src/sitesense/app.py").read_text())
page_inventory = []
for node in ast.walk(navigation):
    if (
        isinstance(node, ast.Call)
        and isinstance(node.func, ast.Attribute)
        and node.func.attr == "Page"
    ):
        values = {
            keyword.arg: keyword.value.value
            for keyword in node.keywords
            if isinstance(keyword.value, ast.Constant)
        }
        source_page = (
            node.args[0].right.value
            if node.args
            and isinstance(node.args[0], ast.BinOp)
            and isinstance(node.args[0].right, ast.Constant)
            else None
        )
        page_inventory.append({"Title": values.get("title"), "Source page": source_page})
display(pd.DataFrame(page_inventory))
save_table(pd.DataFrame(page_inventory), "current_navigation_inventory.csv")

## Results

### 3. Top exact source categories in the parent category

This ranking uses the full parent catalog, **before** applying `SOURCE_CATEGORY`, with total Recorded check-ins as the default ordering. Zero-activity listings remain eligible. A broad/high-volume tag need not indicate the best new-store opportunity: activity concentration, competition, incomplete market coverage and snapshot membership all affect interpretation.

In [ ]:
available_tags = business_analysis.subcategory_groups(PARENT_CATEGORY, catalog.businesses)[
    "Source categories"
]
if SOURCE_CATEGORY is not None and SOURCE_CATEGORY not in available_tags:
    raise ValueError(f"Choose an exact source category from: {available_tags}")
records = repository.load_business_activity(effective) if effective is not None else ()
parent_summaries = (
    business_analysis.summarize_businesses(catalog.businesses, records, covered_start, covered_end)
    if covered
    else ()
)
source_rows = [
    {
        "Source category": row.category,
        "Listed businesses": row.business_count,
        "Businesses with check-ins": row.active_business_count,
        "Recorded check-ins": row.checkin_count,
        "Leader share %": 100 * row.top_business_share
        if row.top_business_share is not None
        else None,
    }
    for row in business_analysis.rank_categories(parent_summaries, available_tags)
]
source_ranking = pd.DataFrame(
    source_rows,
    columns=[
        "Source category",
        "Listed businesses",
        "Businesses with check-ins",
        "Recorded check-ins",
        "Leader share %",
    ],
)
if not covered:
    source_ranking = pd.DataFrame(
        [
            {
                "Source category": tag,
                "Listed businesses": sum(tag in item.categories for item in catalog.businesses),
                "Businesses with check-ins": None,
                "Recorded check-ins": None,
                "Leader share %": None,
            }
            for tag in available_tags
        ]
    )
display(source_ranking.head(TOP_N).round(2))
save_table(source_ranking, "parent_source_category_ranking.csv")
if covered and not source_ranking.empty:
    fig, axis = plt.subplots(figsize=(10, 5), layout="constrained")
    leading = source_ranking.head(TOP_N).iloc[::-1]
    axis.barh(leading["Source category"], leading["Recorded check-ins"], color="#245f93")
    axis.set(
        xlabel="Recorded check-ins",
        title=f"Top overlapping source categories · {CITY} · {PARENT_CATEGORY}",
    )
    save_figure(fig, "source_category_recorded_activity.png")

### 4. Named competitors in the selected niche

A listing belongs to the niche only when its exact source tags contain the selection. Stable business IDs break ties and distinguish duplicate names. The full table includes missing-coordinate listings; a map can display only the leading businesses with valid coordinates. Source categories, stars, reviews and open status are clearly labeled as snapshots.

In [ ]:
scope_businesses = tuple(
    b for b in catalog.businesses if SOURCE_CATEGORY is None or SOURCE_CATEGORY in b.categories
)
if not scope_businesses:
    raise ValueError("No businesses match the exact selected source category.")
by_id = {item.business.business_id: item for item in parent_summaries}
ranked = (
    sorted(scope_businesses, key=lambda b: (-by_id[b.business_id].checkin_count, b.business_id))
    if covered
    else sorted(scope_businesses, key=lambda b: b.business_id)
)
rows = []
for business in ranked:
    summary = by_id.get(business.business_id)
    rows.append(
        {
            "Business ID": business.business_id,
            "Business": business.name,
            "Recorded check-ins": summary.checkin_count if summary else None,
            "Days with check-ins": summary.active_days if summary else None,
            "Weekend share %": 100 * summary.weekend_share
            if summary and summary.weekend_share is not None
            else None,
            "Rating (snapshot)": business.stars,
            "Reviews (snapshot)": business.review_count,
            "Status (snapshot)": "Listed open" if business.is_open else "Listed closed",
            "Source categories (snapshot)": "; ".join(business.categories),
            "Latitude": business.latitude,
            "Longitude": business.longitude,
            "Mapped weather station": business.weather_cell_id,
        }
    )
business_ranking = pd.DataFrame(rows)
display(business_ranking.head(TOP_N).round(2))
save_table(business_ranking, "selected_source_business_ranking.csv")
coordinate_ok = business_ranking.Latitude.notna() & business_ranking.Longitude.notna()
map_preview = business_ranking.loc[coordinate_ok].head(TOP_N)
save_table(map_preview, "top_business_map_points.csv")
print(
    {
        "matching_businesses": len(scope_businesses),
        "map_top_businesses": len(map_preview),
        "businesses_without_coordinates_retained_in_totals": int((~coordinate_ok).sum()),
        "recorded_total": int(business_ranking["Recorded check-ins"].sum()) if covered else None,
    }
)

### 5. Compare two to four businesses: profile and activity patterns

Supply stable IDs in `COMPARE_BUSINESS_IDS`, or compare the two leading competitors in the current niche. The matrix uses the same selected covered dates; ratings, reviews, status and categories retain their snapshot labels. Weather is shared context rather than another copy of each business's observation series.

In [ ]:
selected_ids = tuple(dict.fromkeys(COMPARE_BUSINESS_IDS)) or tuple(
    b.business_id for b in ranked[:2]
)
if not 1 <= len(selected_ids) <= 4 or not set(selected_ids) <= {
    b.business_id for b in scope_businesses
}:
    raise ValueError(
        "Choose one to four unique IDs in the selected niche; use two or more for comparison."
    )
if len(selected_ids) < 2:
    print("Only one matching listing is available; a multi-business comparison is unavailable.")
comparison = business_ranking.set_index("Business ID").loc[list(selected_ids)]
comparison_columns = [
    "Recorded check-ins",
    "Days with check-ins",
    "Weekend share %",
    "Rating (snapshot)",
    "Reviews (snapshot)",
    "Status (snapshot)",
    "Source categories (snapshot)",
]
comparison_matrix = (
    comparison[comparison_columns]
    .rename(index=lambda identifier: f"{comparison.loc[identifier, 'Business']} ({identifier})")
    .T
)
comparison_matrix.index.name = "Metric"
display(comparison_matrix)
save_table(comparison_matrix.reset_index(), "comparison_profile_activity_matrix.csv")
selected_businesses = tuple(
    next(b for b in scope_businesses if b.business_id == identifier) for identifier in selected_ids
)

### 6. Build daily activity and distinct weather evidence

The shared calendar helper chooses week/month/year bins from the selected range. Activity uses **totals in the same clipped calendar bins** as the weather boxes. Covered gaps become zero only after successful reads. Scope activity includes every matching listing and is independent of map limits and comparison selections.

Temperature uses optional representative city-airport hourly reports when imported; otherwise accepted mapped-station daily lows/highs. Rain uses accepted ACIS daily totals. Stations are counted once per report date/variable, never once per business. Missing or rejected reports remain missing; dry and trace rain remain zero-valued observations. Exported boxes include sample counts.

In [ ]:
scope_daily = repository.load_scope_daily(effective) if covered else ()
comparison_daily = {
    b.business_id: repository.load_business_daily(effective, b.business_id) if covered else ()
    for b in selected_businesses
}
if covered:
    assert sum(row.checkin_count for row in scope_daily) == int(
        business_ranking["Recorded check-ins"].sum()
    )
    for business in selected_businesses:
        assert (
            sum(row.checkin_count for row in comparison_daily[business.business_id])
            == by_id[business.business_id].checkin_count
        )
mapped_ids = {b.weather_cell_id for b in scope_businesses}
station_cells = tuple(
    {
        c.weather_cell_id: c
        for c in catalog.weather_cells
        if c.source_kind == "acis_station" and c.weather_cell_id in mapped_ids
    }.values()
)
weather = load_daily_station_weather(station_cells, START_DATE, END_DATE) if station_cells else {}
hourly_samples = (
    repository.load_hourly_weather(CITY, STATE, START_DATE, END_DATE)
    if hourly_revision and WEATHER_VARIABLE == "Temperature"
    else ()
)
hourly_rows = hourly_chart_rows(hourly_samples, "Temperature (°C)")


def weather_rows_for(businesses):
    if hourly_rows:
        return hourly_rows
    station_ids = sorted({b.weather_cell_id for b in businesses if b.weather_cell_id})
    variables = (
        {"pcpn": "Precipitation (mm/day)"}
        if WEATHER_VARIABLE == "Precipitation"
        else {"tmin": "Daily low", "tmax": "Daily high"}
    )
    return [
        {
            "Date": day.isoformat(),
            "Weather value": value,
            "Series": "Precipitation" if WEATHER_VARIABLE == "Precipitation" else "Temperature",
            "Weather station": f"{len(station_ids)} distinct mapped stations",
            "Weather source": "ACIS accepted daily observations",
            "Observation frequency": "Daily summaries",
        }
        for identifier in station_ids
        for day, reports in weather.get(identifier, {}).items()
        for variable in variables
        if (value := reports.get(variable)) is not None
    ]


def period_activity(records):
    rows = [
        {"Date": row.activity_date.isoformat(), "Recorded check-ins": row.checkin_count}
        for row in records
    ]
    return activity_period_rows(
        rows, START_DATE, END_DATE, coverage=(coverage.start_date, coverage.end_date)
    )


scope_periods = period_activity(scope_daily)
comparison_periods = {
    identifier: period_activity(records) for identifier, records in comparison_daily.items()
}
save_table(pd.DataFrame(scope_periods), "scope_recorded_calendar_totals.csv")
save_table(
    pd.DataFrame(
        [
            dict(row, business_id=identifier)
            for identifier, rows in comparison_periods.items()
            for row in rows
        ]
    ),
    "comparison_recorded_calendar_totals.csv",
)
print(
    {
        "calendar_grain": calendar_grain(START_DATE, END_DATE),
        "scope_stations": len(station_cells),
        "hourly_temperature_samples": len(hourly_rows),
        "scope_activity_reconciled": covered,
    }
)

### 7. Weather and recorded activity — all matching businesses and comparison

The first chart mirrors the Location map's all-business scope. The second overlays individual activity lines for the comparison IDs, with one shared weather distribution. Orange boxes span Q1–Q3, their line is the median, whiskers show observed min/max, and the diamond marks the mean. Weather and activity use separate units and axes; visual co-movement does not establish causality.

In [ ]:
def combined_chart(activity_series, businesses, title, filename):
    weather_rows = weather_rows_for(businesses)
    boxes = weather_boxplot_display_rows(
        weather_boxplot_rows(
            weather_rows,
            START_DATE,
            END_DATE,
            combine_temperature=WEATHER_VARIABLE == "Temperature",
            align_calendar_periods=True,
        )
    )
    fig, activity_axis = plt.subplots(figsize=(13, 5), layout="constrained")
    weather_axis = activity_axis.twinx()
    if boxes:
        stats = [
            dict(
                label=row["Period"],
                whislo=row["Minimum"],
                q1=row["Q1"],
                med=row["Median"],
                q3=row["Q3"],
                whishi=row["Maximum"],
                mean=row["Mean"],
                fliers=[],
            )
            for row in boxes
        ]
        positions = mdates.date2num(
            [pd.Timestamp(row["Weather period"]).to_pydatetime() for row in boxes]
        )
        period_count = max(1, len(period_activity(())))
        width = max(1, (END_DATE - START_DATE).days / period_count * 0.3)
        weather_axis.bxp(
            stats,
            positions=positions,
            widths=width,
            manage_ticks=False,
            showmeans=True,
            patch_artist=True,
            boxprops={"facecolor": "#efb880", "alpha": 0.35},
            medianprops={"color": "#b85c18"},
            meanprops={
                "marker": "D",
                "markerfacecolor": "#e67e22",
                "markeredgecolor": "white",
                "markersize": 5,
            },
        )
    for series_number, (label, periods) in enumerate(activity_series.items()):
        activity_axis.plot(
            pd.to_datetime([row["Period midpoint"] for row in periods]),
            [row["Recorded check-ins"] for row in periods],
            marker="o",
            label=label,
            color=["#245f93", "#7b48a2", "#26816e", "#a83a4c"][series_number % 4],
        )
    activity_axis.set(
        title=title,
        xlabel=f"Calendar {calendar_grain(START_DATE, END_DATE)}; same selected-date bins",
        ylabel="Recorded check-ins",
    )
    weather_axis.set_ylabel(
        "Precipitation (mm/day)" if WEATHER_VARIABLE == "Precipitation" else "Temperature (°C)"
    )
    activity_axis.set_ylim(bottom=0)
    activity_axis.legend(loc="upper left", fontsize=8)
    activity_axis.grid(alpha=0.15)
    if not boxes:
        activity_axis.text(
            0.5,
            0.95,
            "Accepted weather observations unavailable",
            transform=activity_axis.transAxes,
            ha="center",
        )
    save_figure(fig, filename + ".png")
    save_table(pd.DataFrame(boxes), filename + "_weather_boxes.csv")
    return boxes


scope_boxes = combined_chart(
    {f"All {len(scope_businesses)} matching businesses": scope_periods},
    scope_businesses,
    "Weather and recorded activity · all matching businesses",
    "scope_weather_recorded_activity",
)
comparison_boxes = combined_chart(
    {
        f"{business.name} ({business.business_id})": comparison_periods[business.business_id]
        for business in selected_businesses
    },
    selected_businesses,
    "Weather and recorded activity · selected competitors",
    "comparison_weather_recorded_activity",
)
display(
    pd.DataFrame(scope_boxes)[
        [
            "Period",
            "Weather source",
            "Observation frequency",
            "Observed readings",
            "Observed days",
            "Minimum",
            "Median",
            "Mean",
            "Maximum",
        ]
    ].head(8)
    if scope_boxes
    else pd.DataFrame({"Status": ["Weather unavailable"]})
)

### 8. Supported HGB weather-scenario projections after recorded coverage

The current app serves the verified `models/business_hgb` artifact. For a reviewable preparation workflow, this audit first tries the latest complete notebook 04 run's `hgb_artifact`, then the installed app bundle. Prepared artifacts require a complete outer run manifest and unchanged declared file checksums, followed by the same production `business_hgb.load_bundle` and `project` functions as the app. The loader checks import revision, source code, lockfile, package versions and payload hashes. A prepared bundle is explicitly labeled **unpublished**; reading it does not replace the installed app files. An unavailable HGB artifact remains unavailable; a calendar reference is never silently substituted. [Notebook 04](04_full_history_forecast_training.ipynb) separately documents calendar baseline training and explicit model preparation.

`WEATHER_SCENARIO = None` uses each business's training-station/month averages from 2019–2021. A custom `(low °C, high °C, rain mm/day)` tuple applies those same conditions on every selected day. Both are **conditional scenarios, not live future weather forecasts**. HGB is served for weather scenarios without claiming it beats the calendar baseline on held-out accuracy.

The ranking estimates **supported existing businesses at their recorded coordinates**. A source-category total covers only its supported subset and is labeled with modeled/catalog counts. A higher projection does not establish the best new niche, a vacant site, profitability, causal weather sensitivity or new-shop activity. No model is trained or published by this audit.

In [ ]:
candidates = []
for run in sorted((ROOT / "data/processed/business_forecast_training").glob("run_*"), reverse=True):
    try:
        run_manifest = json.loads((run / "manifest.json").read_bytes())
        hashes = run_manifest.get("files_sha256", {})
        valid = (
            run_manifest.get("status") == "complete"
            and run_manifest.get("hgb_artifact_directory") == "hgb_artifact"
            and run_manifest.get("app_hgb_status") == "verified"
            and bool(hashes)
            and {"hgb_artifact/manifest.json", "hgb_artifact/models.joblib"} <= set(hashes)
            and all(
                (run / name).is_file() and sha256(run / name) == digest
                for name, digest in hashes.items()
            )
        )
    except (OSError, ValueError, TypeError):
        valid = False
    if valid:
        candidates.append((run / "hgb_artifact", "Prepared HGB · unpublished"))
candidates.append((MODEL_DIRECTORY, "Installed app HGB"))
bundle, bundle_path, artifact_kind = None, None, None
bundle_checks = []
for candidate, kind in candidates:
    try:
        candidate_bundle = business_hgb.load_bundle(candidate, import_revision)
    except ValueError:
        bundle_checks.append(
            {
                "Artifact": str(candidate.relative_to(ROOT)),
                "Kind": kind,
                "Status": "HGB unavailable or incompatible",
            }
        )
        continue
    bundle, bundle_path, artifact_kind = candidate_bundle, candidate, kind
    bundle_checks.append(
        {"Artifact": str(candidate.relative_to(ROOT)), "Kind": kind, "Status": "Verified HGB"}
    )
    break
display(pd.DataFrame(bundle_checks))
weather_basis = (
    "Training-station/month averages from 2019–2021; seasonal scenario"
    if WEATHER_SCENARIO is None
    else f"Custom scenario: low/high/rain = {WEATHER_SCENARIO}; same weather each day"
)
print("Weather basis:", weather_basis, "· not a live weather forecast")
forecast_table = pd.DataFrame()
forecast_categories = pd.DataFrame()
if bundle is None:
    print("App HGB projections unavailable. Prepare the model explicitly outside this audit:")
    print("uv run --frozen --env-file .env python scripts/train_business_hgb.py")
else:
    current_ids = tuple(b.business_id for b in scope_businesses)
    projected = business_hgb.project(
        bundle,
        CITY,
        STATE,
        PARENT_CATEGORY,
        SOURCE_CATEGORY,
        FUTURE_START,
        FUTURE_END,
        current_ids,
        WEATHER_SCENARIO,
    )
    assert {row.group.business_id for row in projected} <= set(current_ids)
    forecast_table = pd.DataFrame(
        [
            {
                "Business ID": row.group.business_id,
                "Business": row.group.name,
                "Projected check-ins": row.period_rate,
                "Calendar days": row.calendar_days,
                "Latitude": row.group.latitude,
                "Longitude": row.group.longitude,
                "Source categories (snapshot)": "; ".join(row.group.source_categories),
                "Weather scenario": weather_basis,
            }
            for row in projected
        ]
    )
    print(
        {
            "artifact": str(bundle_path.relative_to(ROOT)),
            "artifact_kind": artifact_kind,
            "model": bundle.manifest["model_name"],
            "modeled_matching_businesses": len(projected),
            "matching_catalog_businesses": len(scope_businesses),
            "unmodeled_businesses": len(scope_businesses) - len(projected),
        }
    )
    display(forecast_table.head(TOP_N).round(2))
    parent_ids = tuple(b.business_id for b in catalog.businesses)
    parent_projection = business_hgb.project(
        bundle,
        CITY,
        STATE,
        PARENT_CATEGORY,
        None,
        FUTURE_START,
        FUTURE_END,
        parent_ids,
        WEATHER_SCENARIO,
    )
    assert {row.group.business_id for row in parent_projection} <= set(parent_ids)
    category_projection_rows = []
    for tag in available_tags:
        members = [row for row in parent_projection if tag in row.group.source_categories]
        category_projection_rows.append(
            {
                "Source category": tag,
                "Catalog businesses": sum(tag in b.categories for b in catalog.businesses),
                "Modeled businesses": len(members),
                "Projected check-ins (supported subset)": sum(row.period_rate for row in members)
                if members
                else None,
                "Weather scenario": weather_basis,
            }
        )
    forecast_categories = pd.DataFrame(category_projection_rows).sort_values(
        ["Projected check-ins (supported subset)", "Source category"],
        ascending=[False, True],
        na_position="last",
    )
    display(forecast_categories.head(TOP_N).round(2))
    save_table(forecast_table, "supported_business_hgb_future_scenario.csv")
    save_table(forecast_categories, "source_category_hgb_future_supported_subsets.csv")
save_table(pd.DataFrame(bundle_checks), "forecast_artifact_verification.csv")

### 9. Rebuilt individual-business experiment evidence

The app's served HGB bundle, notebook 04's calendar reference and the HGB/GAM/CatBoost experiments are separate artifacts. Only new individual-business runs from notebooks 03 and 05 may contribute comparison evidence here. The run loader verifies its manifest; unavailable or incompatible research is reported explicitly. No frozen ZIP-model metrics are substituted. Observed-weather experiments must remain labeled retrospective conditional predictions unless inputs were available at the forecast origin.

In [ ]:
research_sources = [
    ROOT / "data/processed/business_forecast_poc",
    ROOT / "data/processed/business_model_comparison",
]
research_evidence = []
try:
    from sitesense.business_research import latest_run, load_run
except ImportError:
    print(
        "The individual-business experiment helper is unavailable; "
        "no research metrics are included."
    )
else:
    for directory in research_sources:
        try:
            run_path = latest_run(directory)
            result = load_run(run_path)
            if result.manifest.get("import_revision") != import_revision:
                raise ValueError("Research import differs from this audit's current import.")
        except (OSError, ValueError, KeyError, TypeError):
            print(
                f"No compatible complete individual-business run in {directory.relative_to(ROOT)}"
            )
            continue
        config = result.manifest["config"]
        same_niche = (
            config["city"],
            config["state"],
            config["category"],
            config["source_category"],
        ) == (CITY, STATE, PARENT_CATEGORY, SOURCE_CATEGORY)
        evidence_scope = {
            "City": config["city"],
            "State": config["state"],
            "Parent": config["category"],
            "Source category": config["source_category"] or "All source categories",
            "Maximum sampled businesses per fit": config["max_businesses"],
            "Matches audit niche": same_niche,
            "Run": str(result.directory.relative_to(ROOT)),
        }
        display(pd.DataFrame([evidence_scope]))
        if not same_niche:
            print(
                "Separate research cohort: these errors do not validate the "
                "selected audit niche or its app projections."
            )
        metric_table = result.metrics.assign(
            research_city=config["city"],
            research_state=config["state"],
            research_parent=config["category"],
            research_source_category=config["source_category"],
            maximum_sampled_businesses=config["max_businesses"],
        )
        display(metric_table)
        save_table(metric_table, directory.name + "_verified_metrics.csv")
        research_evidence.append(
            {
                "root": str(directory.relative_to(ROOT)),
                "run": str(result.directory.relative_to(ROOT)),
                "manifest_sha256": sha256(result.directory / "manifest.json"),
                "config": config,
                "import_revision": result.manifest["import_revision"],
                "matches_audit_niche": same_niche,
            }
        )

### 10. Reconciliation, current file fingerprints and audit manifest

Check stable IDs, source membership, exact totals, consistent calendar bins and the unchanged import revision. The manifest fingerprints current source files and generated artifacts. It contains no database URL, credentials, raw source records, prior WBS completion labels or frozen line-number citations.

In [ ]:
assert len({b.business_id for b in scope_businesses}) == len(scope_businesses)
assert all(SOURCE_CATEGORY is None or SOURCE_CATEGORY in b.categories for b in scope_businesses)
assert all(PARENT_CATEGORY in b.categories for b in scope_businesses)
assert set(selected_ids) <= {b.business_id for b in scope_businesses}
if covered:
    assert sum(row["Recorded check-ins"] for row in scope_periods) == int(
        business_ranking["Recorded check-ins"].sum()
    )
    assert all(row["Covered calendar days"] > 0 for row in scope_periods)
    for periods in comparison_periods.values():
        assert [(row["Period start"], row["Period end"]) for row in periods] == [
            (row["Period start"], row["Period end"]) for row in scope_periods
        ]
assert repository.get_import_revision() == import_revision, "Import changed; rerun the audit."
assert repository.get_hourly_weather_revision() == hourly_revision, (
    "Hourly import changed; rerun the audit."
)
SOURCE_HASHES_AT_END = {
    name: sha256(ROOT / name) for name in SOURCE_PATHS if (ROOT / name).is_file()
}
changed_during_run = [
    name
    for name, digest in SOURCE_HASHES_AT_START.items()
    if SOURCE_HASHES_AT_END.get(name) != digest
]
if changed_during_run:
    print(
        "Source files changed during this audit; review the captured hashes "
        "and rerun for a consistent code snapshot:",
        changed_during_run,
    )
manifest = {
    "schema_version": 1,
    "status": "complete",
    "executed_at_utc": datetime.now(UTC).isoformat(),
    "grain": "business_id_parent_day",
    "target": "recorded_checkins",
    "read_only_database": readonly == "on",
    "import_revision": import_revision,
    "hourly_revision": hourly_revision,
    "activity_coverage": asdict(coverage),
    "parameters": {
        "city": CITY,
        "state": STATE,
        "parent_category": PARENT_CATEGORY,
        "source_category": SOURCE_CATEGORY,
        "start": START_DATE.isoformat(),
        "end": END_DATE.isoformat(),
        "future_start": FUTURE_START.isoformat(),
        "future_end": FUTURE_END.isoformat(),
        "weather_scenario": WEATHER_SCENARIO,
        "weather_basis": weather_basis,
        "app_forecast_model": "hgb",
        "comparison_business_ids": selected_ids,
        "weather_variable": WEATHER_VARIABLE,
    },
    "versions": {
        "python": platform.python_version(),
        **{p: version(p) for p in ("pandas", "matplotlib", "psycopg", "scikit-learn")},
    },
    "source_sha256_at_start": SOURCE_HASHES_AT_START,
    "source_sha256_at_end": SOURCE_HASHES_AT_END,
    "source_changed_during_run": changed_during_run,
    "navigation": page_inventory,
    "forecast_artifact": str(bundle_path.relative_to(ROOT)) if bundle_path else None,
    "forecast_manifest_sha256": sha256(bundle_path / "manifest.json") if bundle_path else None,
    "forecast_artifact_kind": artifact_kind,
    "research_evidence": research_evidence,
    "checks": {
        "identity_and_scope": "passed",
        "source_import_unchanged": "passed",
        "recorded_totals_and_calendar_bins": "passed"
        if covered
        else "no covered activity interval",
    },
    "artifacts": [{"path": name, "sha256": sha256(RUN_DIR / name)} for name in ARTIFACTS],
    "limits": [
        "recorded check-ins are not customers or revenue",
        "snapshot source tags overlap",
        "uncovered/failed activity is unknown",
        "station/activity time alignment unverified",
        "weather is descriptive, not causal",
        "projections cover supported existing businesses only",
        "no new-store success estimate or vacant-site recommendation",
    ],
}
(RUN_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2, default=str) + "\n")
print(
    {
        "status": "complete",
        "artifacts": len(ARTIFACTS),
        "manifest": str((RUN_DIR / "manifest.json").relative_to(ROOT)),
    }
)

## Takeaways

Use the executed source-category table to see which exact tags have the most recorded activity in the selected parent cohort, then inspect named competitors and the profile/activity comparison. Interpret weather boxes and activity totals on the same periods, while checking missing samples and unverified day alignment.

For future dates, compare only businesses explicitly supported by a verified bundle and inspect the modeled/catalog denominator for each source category. Choosing a new niche or shop address requires further evidence such as rents, vacancies, access, demographics, capacity, demand coverage and costs; this audit does not estimate a new store's performance.

Refresh the database import or model preparation explicitly outside the notebook when required, then rerun the full audit. The resulting manifest records current evidence rather than a copied historical project assessment.